# 05 — Validate And Prepare Analysis Datasets

## Research purpose

Validate the facility database and prepare facility and campus analysis tables from the estimated capacities.


## Imports

Load the libraries for table processing, numerical validation and copying the data dictionary.


In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

## Project paths

Locate the project folder and define the paths for the estimated facilities, manual data dictionary and final tables.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_FINAL = PROJECT_ROOT / "data" / "final"

FACILITY_OUTPUT = DATA_FINAL / "hyperscale_facilities_analysis.csv"
CAMPUS_OUTPUT = DATA_FINAL / "hyperscale_campuses_analysis.csv"
manual_dictionary_path = DATA_MANUAL / "facility_data_dictionary.csv"

## Facility schema

Define the 35 fields in the final database, the required input columns and the accepted confidence labels. Retain both the IANA timezone and the fixed-reference UTC offset.


In [3]:
FINAL_COLUMNS = [
    "facility_id",
    "facility_name",
    "operator_group",
    "latitude",
    "longitude",
    "country",
    "country_code",
    "world_region",
    "sub_region",
    "timezone",
    "utc_offset",
    "campus_group_id",
    "campus_group_size",
    "capacity_record_role",
    "site_area_m2",
    "building_footprint_area_m2",
    "height_m",
    "height_source",
    "gross_floor_area_m2",
    "floors_assumed",
    "floor_count_method",
    "floor_count_confidence",
    "footprint_method",
    "footprint_confidence",
    "capacity_mw",
    "it_load_mw",
    "capacity_estimation_method",
    "capacity_confidence",
    "assumed_pue",
    "status",
    "source_type",
    "source_ids",
    "osm_id",
    "osm_type",
    "notes",
]

REQUIRED_INPUT_COLUMNS = (
    set(FINAL_COLUMNS) - {"facility_id", "facility_name", "status"}
) | {"facility_name_raw", "operator_standardized"}

CONFIDENCE_ALLOWED_VALUES = {
    "floor_count_confidence": {"high", "medium", "low", "unknown"},
    "footprint_confidence": {"medium", "low", "unknown", "not_applicable"},
    "capacity_confidence": {"medium", "low", "unknown"},
}

## Load estimated facilities

Read the capacity estimates from notebook 04 and check that all required scientific fields are present.


In [4]:
df = pd.read_csv(DATA_PROCESSED / "datacenters_hyperscalers_capacity_estimated.csv")
missing = sorted(REQUIRED_INPUT_COLUMNS - set(df.columns))

if missing:
    raise ValueError(f"Estimated input is missing scientific fields: {missing}")

print(f"Estimated rows: {len(df):,}")

Estimated rows: 511


## Prepare the facility database

Assign facility names and country/operator identifiers, and treat mapped OSM infrastructure as operational in the baseline. Retain rows with both capacity and IT-load estimates while recording exclusion reasons for missing estimates and context-only polygons. These estimates remain capacity proxies derived in notebook 04.


In [5]:
df["facility_name"] = df["facility_name_raw"].fillna(df["operator_standardized"])
df["status"] = df["status"] if "status" in df.columns else pd.NA

# OSM-derived rows represent mapped current infrastructure in the thesis baseline.
# Treat them as operational in this baseline; OSM is mapped physical evidence
# rather than an operator-reported operating certificate.
if "source_type" in df.columns:
    osm_source_mask = df["source_type"].astype("string").str.lower().eq("osm")
    df.loc[osm_source_mask, "status"] = "operational"


# Assign sequential country/operator IDs using the documented facility sort order.
def make_facility_ids(frame):
    result = frame.copy()
    result["country_code"] = result["country_code"].fillna("XXX")
    ids = []
    counters = {}
    for _, row in result.sort_values(["country_code", "operator_group", "facility_name"]).iterrows():
        key = (row["country_code"], row["operator_group"])
        counters[key] = counters.get(key, 0) + 1
        ids.append((row.name, f"{row['country_code']}-{str(row['operator_group']).upper()}-{counters[key]:04d}"))
    id_series = pd.Series(index=result.index, dtype=object)
    for idx, facility_id in ids:
        id_series.loc[idx] = facility_id
    result["facility_id"] = id_series
    return result


# Describe the missing-capacity and context-role reasons for excluded records.
def excluded_reason(row):
    reasons = []
    if row.get("capacity_record_role") == "site_context_only":
        reasons.append("site_context_only_capacity_represented_by_building_rows")
    if pd.isna(row.get("capacity_mw")) or pd.isna(row.get("it_load_mw")):
        reasons.append("missing_capacity_or_it_load")
    return ";".join(reasons) if reasons else "not_excluded"


df = make_facility_ids(df)

capacity_bearing_mask = (
    df["capacity_mw"].notna()
    & df["it_load_mw"].notna()
    & df["capacity_record_role"].ne("site_context_only")
)

excluded = df.loc[~capacity_bearing_mask, FINAL_COLUMNS].copy()
excluded["exclusion_reason"] = excluded.apply(excluded_reason, axis=1)

final = df.loc[capacity_bearing_mask, FINAL_COLUMNS].copy()

print(f"Capacity-bearing final rows: {len(final):,}")
print(f"Excluded/review rows: {len(excluded):,}")
final.head()

Capacity-bearing final rows: 489
Excluded/review rows: 22


,facility_id,facility_name,operator_group,latitude,longitude,country,country_code,world_region,sub_region,timezone,...,it_load_mw,capacity_estimation_method,capacity_confidence,assumed_pue,status,source_type,source_ids,osm_id,osm_type,notes
0,USA-GOOGLE-0061,Google Fiber,Google,33.771568,-84.366859,United States,USA,North America,Northern America,America/New_York,...,42.913205,footprint_x_floors_x_it_space_share_x_power_de...,low,1.09,operational,osm,osm_node_4879481820,4879481820,node,NaN
1,SGP-AMAZON-0002,Amazon SIN4,Amazon,1.335397,103.890335,Singapore,SGP,Asia Pacific,South-eastern Asia,Asia/Singapore,...,53.602780,footprint_x_floors_x_it_space_share_x_power_de...,low,1.15,operational,osm,osm_node_8208591051,8208591051,node,NaN
3,MYS-GOOGLE-0001,Google Singapore DC,Google,1.351150,103.709178,Malaysia,MYS,Asia Pacific,South-eastern Asia,Asia/Singapore,...,39.328882,footprint_x_floors_x_it_space_share_x_power_de...,low,1.09,operational,osm,osm_node_8208591112,8208591112,node,NaN
4,SGP-AMAZON-0001,Amazon SIN3,Amazon,1.340530,103.944283,Singapore,SGP,Asia Pacific,South-eastern Asia,Asia/Singapore,...,34.569969,footprint_x_floors_x_it_space_share_x_power_de...,low,1.15,operational,osm,osm_node_8208598531,8208598531,node,NaN
5,SGP-AMAZON-0003,Amazon SIN5,Amazon,1.336682,103.893767,Singapore,SGP,Asia Pacific,South-eastern Asia,Asia/Singapore,...,26.768030,footprint_x_floors_x_it_space_share_x_power_de...,low,1.15,operational,osm,osm_node_8208598665,8208598665,node,NaN


## Validate the facility records

Check that the manual data dictionary matches the final schema and summarize records by operator and source. Validate identifiers, coordinates, capacities and confidence labels, and stop if any checks fail. Display the validation report in the notebook.


In [6]:
data_dictionary = pd.read_csv(manual_dictionary_path)
if data_dictionary["field"].tolist() != final.columns.tolist():
    missing_dictionary = sorted(set(final.columns) - set(data_dictionary["field"]))
    extra_dictionary = sorted(set(data_dictionary["field"]) - set(final.columns))
    raise ValueError(f"Data dictionary/schema mismatch; missing={missing_dictionary}, extra={extra_dictionary}")
source_summary = final.groupby(["operator_group", "source_type"], dropna=False).size().reset_index(name="record_count")

issues = []
confidence_audit = pd.concat([final, excluded], ignore_index=True, sort=False)
for field, allowed in CONFIDENCE_ALLOWED_VALUES.items():
    observed = set(confidence_audit[field].dropna().astype("string").str.strip())
    invalid = sorted(observed - allowed)
    if invalid:
        issues.append(f"Invalid {field} values: {invalid}")
if final["facility_id"].duplicated().any(): issues.append("Duplicate facility IDs")
if not final["latitude"].between(-90, 90).all(): issues.append("Invalid latitude")
if not final["longitude"].between(-180, 180).all(): issues.append("Invalid longitude")
if final[["capacity_mw", "it_load_mw"]].isna().any().any(): issues.append("Missing capacity")
if (final[["capacity_mw", "it_load_mw"]] < 0).any().any(): issues.append("Negative capacity")

report = ["Hyperscale facility validation report", f"Final rows: {len(final):,}", f"Excluded rows: {len(excluded):,}", "Issues:", *(issues or ["No blocking validation issues detected."])]
if issues: raise ValueError(issues)
print("\n".join(report))

Hyperscale facility validation report
Final rows: 489
Excluded rows: 22
Issues:
No blocking validation issues detected.


## Prepare the facility analysis table

Select and rename the fields needed for facility analysis, convert numeric columns and sort the records. Check that capacity and IT load are available for every retained facility.


In [7]:
facility_column_map = {
    "facility_id": "facility_id",
    "operator_group": "operator_group",
    "latitude": "lat",
    "longitude": "lon",
    "country": "country",
    "country_code": "country_code",
    "world_region": "world_region",
    "sub_region": "sub_region",
    "timezone": "timezone",
    "utc_offset": "utc_offset",
    "campus_group_id": "campus_group",
    "campus_group_size": "campus_group_size",
    "gross_floor_area_m2": "gross_floor_area_m2",
    "floors_assumed": "floors_assumed",
    "capacity_mw": "capacity_mw",
    "it_load_mw": "it_load_mw",
    "assumed_pue": "pue",
    "status": "status",
}

facility_required = set(facility_column_map) | {"capacity_record_role"}
missing = sorted(facility_required - set(final.columns))
if missing:
    raise ValueError(f"Missing required final dataset columns: {missing}")

facilities = final[list(facility_column_map)].rename(columns=facility_column_map)

numeric_columns = [
    "lat",
    "lon",
    "utc_offset",
    "campus_group_size",
    "gross_floor_area_m2",
    "floors_assumed",
    "capacity_mw",
    "it_load_mw",
    "pue",
]
for column in numeric_columns:
    facilities[column] = pd.to_numeric(facilities[column], errors="coerce")

facilities = facilities.sort_values(
    ["operator_group", "country_code", "campus_group", "facility_id"],
    na_position="last",
).reset_index(drop=True)

missing_mw = facilities[facilities["capacity_mw"].isna() | facilities["it_load_mw"].isna()]
if not missing_mw.empty:
    raise ValueError(f"Facility analysis contains {len(missing_mw):,} rows with missing capacity or IT load")

facilities.head()

,facility_id,operator_group,lat,lon,country,country_code,world_region,sub_region,timezone,utc_offset,campus_group,campus_group_size,gross_floor_area_m2,floors_assumed,capacity_mw,it_load_mw,pue,status
0,ARE-AMAZON-0001,Amazon,24.893127,55.109898,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,CAMPUS-00107,1,9840.791967,1.0,20.370439,17.713426,1.15,operational
1,ARE-AMAZON-0002,Amazon,24.268809,54.463322,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,CAMPUS-00161,1,9279.296715,1.0,19.208144,16.702734,1.15,operational
2,ARE-AMAZON-0003,Amazon,24.708930,54.741704,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,CAMPUS-00162,1,10196.728075,1.0,21.107227,18.354111,1.15,operational
3,AUS-AMAZON-0001,Amazon,-34.037120,150.768177,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,10.0,CAMPUS-00068,2,23358.962191,2.0,48.353052,42.046132,1.15,operational
4,AUS-AMAZON-0003,Amazon,-34.035751,150.765894,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,10.0,CAMPUS-00068,2,8645.734091,1.0,17.896670,15.562321,1.15,operational


## Define campus aggregation

Prepare campus grouping keys and define helpers for combining text fields, status, coordinates and capacity totals. Leave entirely missing values undefined and calculate campus PUE from total power divided by IT load, with mean recorded PUE as a fallback.


In [8]:
work = final.copy()

numeric_source_columns = [
    "latitude",
    "longitude",
    "utc_offset",
    "campus_group_size",
    "gross_floor_area_m2",
    "capacity_mw",
    "it_load_mw",
    "assumed_pue",
]
for column in numeric_source_columns:
    work[column] = pd.to_numeric(work[column], errors="coerce")

campus_group = work["campus_group_id"].astype("string").str.strip()
facility_fallback = work["facility_id"].astype("string").str.strip()
work["campus_group_key"] = campus_group.mask(campus_group.isna() | campus_group.eq(""), facility_fallback)


# Retain the first nonblank text value in the current group order.
def first_non_null(series):
    values = series.dropna()
    if values.empty:
        return np.nan
    values = values.astype(str).str.strip()
    values = values[values.ne("")]
    return values.iloc[0] if not values.empty else np.nan


# Calculate a numeric mean while leaving entirely missing groups undefined.
def mean_numeric(series):
    values = pd.to_numeric(series, errors="coerce")
    return values.mean() if values.notna().any() else np.nan


# Sum available numeric values without turning an entirely missing group into zero.
def safe_sum(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values.sum() if not values.empty else np.nan


# Report a shared status or label a group with differing statuses as mixed.
def compact_status(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values.ne("")]
    if values.empty:
        return np.nan
    unique = sorted(values.unique())
    return unique[0] if len(unique) == 1 else "mixed"


# Retain the recorded group size or fall back to the number of accepted rows.
def campus_group_size(group):
    values = pd.to_numeric(group["campus_group_size"], errors="coerce").dropna()
    if not values.empty:
        return int(values.max())
    return int(len(group))


# Combine a campus's attributes, capacity totals, and aggregate PUE.
def aggregate_campus(group):
    gross_floor_area_m2 = safe_sum(group["gross_floor_area_m2"])
    capacity_mw = safe_sum(group["capacity_mw"])
    it_load_mw = safe_sum(group["it_load_mw"])

    if pd.notna(capacity_mw) and pd.notna(it_load_mw) and it_load_mw > 0:
        pue = capacity_mw / it_load_mw
    else:
        pue = mean_numeric(group["assumed_pue"])

    return {
        "campus_group": first_non_null(group["campus_group_key"]),
        "operator_group": first_non_null(group["operator_group"]),
        "lat": mean_numeric(group["latitude"]),
        "lon": mean_numeric(group["longitude"]),
        "country": first_non_null(group["country"]),
        "country_code": first_non_null(group["country_code"]),
        "world_region": first_non_null(group["world_region"]),
        "sub_region": first_non_null(group["sub_region"]),
        "timezone": first_non_null(group["timezone"]),
        "utc_offset": mean_numeric(group["utc_offset"]),
        "campus_group_size": campus_group_size(group),
        "gross_floor_area_m2": gross_floor_area_m2,
        "capacity_mw": capacity_mw,
        "it_load_mw": it_load_mw,
        "pue": pue,
        "status": compact_status(group["status"]),
    }

## Aggregate campus records

Apply the aggregation to each campus group and sort the resulting table by operator, country and campus. Display a sample to inspect the combined records.


In [9]:
campus_records = [
    aggregate_campus(group)
    for _, group in work.groupby("campus_group_key", dropna=False, sort=True)
]
campuses = pd.DataFrame(campus_records)

campuses = campuses.sort_values(
    ["operator_group", "country_code", "campus_group"],
    na_position="last",
).reset_index(drop=True)

campuses.head()

,campus_group,operator_group,lat,lon,country,country_code,world_region,sub_region,timezone,utc_offset,campus_group_size,gross_floor_area_m2,capacity_mw,it_load_mw,pue,status
0,CAMPUS-00107,Amazon,24.893127,55.109898,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,1,9840.791967,20.370439,17.713426,1.15,operational
1,CAMPUS-00161,Amazon,24.268809,54.463322,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,1,9279.296715,19.208144,16.702734,1.15,operational
2,CAMPUS-00162,Amazon,24.708930,54.741704,United Arab Emirates,ARE,Middle East and Africa,Western Asia,Asia/Dubai,4.0,1,10196.728075,21.107227,18.354111,1.15,operational
3,CAMPUS-00068,Amazon,-34.036436,150.767035,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,10.0,2,32004.696281,66.249721,57.608453,1.15,operational
4,CAMPUS-00094,Amazon,-33.795005,150.869881,Australia,AUS,Asia Pacific,Australia and New Zealand,Australia/Sydney,10.0,2,28189.274017,58.351797,50.740693,1.15,operational


## Reconcile capacity totals

Check that facility and campus capacity totals agree within the numerical tolerance. Compare row counts and total MW across the final facility database and both analysis tables.


In [10]:
facility_total = facilities["capacity_mw"].sum()
campus_total = campuses["capacity_mw"].sum()

if abs(facility_total - campus_total) > 1e-9:
    raise ValueError("Facility and campus capacity do not reconcile")

print(pd.DataFrame({"dataset": ["final facilities", "analysis facilities", "analysis campuses"], "rows": [len(final), len(facilities), len(campuses)], "capacity_mw": [final["capacity_mw"].sum(), facility_total, campus_total]}).to_string(index=False))

            dataset  rows  capacity_mw
   final facilities   489 19418.918276
analysis facilities   489 19418.918276
  analysis campuses   164 19418.918276


## Save validated tables

Save the facility database, excluded records, source summary and both analysis tables. Copy the manual data dictionary alongside the validated results.


In [11]:
excluded.to_csv(DATA_FINAL / "hyperscale_facilities_excluded_review.csv", index=False)
final.to_csv(DATA_FINAL / "hyperscale_facilities_final.csv", index=False)
shutil.copyfile(manual_dictionary_path, DATA_FINAL / "hyperscale_facilities_data_dictionary.csv")
source_summary.to_csv(DATA_FINAL / "hyperscale_facilities_source_summary.csv", index=False)
facilities.to_csv(FACILITY_OUTPUT, index=False)
campuses.to_csv(CAMPUS_OUTPUT, index=False)